# TT-LLM: Compression + SFT + KD + Evaluation

Compresses SmolLM2-360M using TT-LLM or SoLA, then:
1. **Stage 1**: Supervised Fine-Tuning (SFT) on Dolly-15K
2. **Stage 2**: Knowledge Distillation (KD) with the uncompressed teacher
3. **Evaluation**: 5 benchmarks (MMLU, HellaSwag, ARC-Easy, PIQA, WinoGrande)

All logic lives in `tt_llm/pipeline.py` — to make changes, just `git pull`.

## 0. Setup

In [ ]:
import os

if not os.path.exists('/content/TT_LLM'):
    !git clone https://github.com/WaelRabah/TT_LLM.git /content/TT_LLM

%cd /content/TT_LLM
!git pull
!pip install -e ".[colab]" -q

# If the repo is private, use a GitHub token instead:
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')
# !git clone https://{token}@github.com/WaelRabah/TT_LLM.git /content/TT_LLM
# %cd /content/TT_LLM
# !pip install -e ".[colab]" -q

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Config

Change these and re-run everything below:

In [ ]:
METHOD = 'sola'         # 'tt_llm' or 'sola'
COMPRESSION_PCT = 30     # remove 30% of params
SFT_EPOCHS = 3
KD_EPOCHS = 2
EVAL_LIMIT = 500         # examples per benchmark (None = full)

## 2. Run Full Pipeline

One call does everything: load → compress → SFT → KD → evaluate.

In [ ]:
from tt_llm.pipeline import run_pipeline

results = run_pipeline(
    method=METHOD,
    compression_pct=COMPRESSION_PCT,
    sft_epochs=SFT_EPOCHS,
    kd_epochs=KD_EPOCHS,
    eval_limit=EVAL_LIMIT,
)

## 3. Results

In [ ]:
import matplotlib.pyplot as plt

# Plot SFT loss
sft = results.get('sft_metrics', {})
if sft.get('train_loss'):
    plt.figure(figsize=(10, 4))
    plt.plot(sft['train_loss'], alpha=0.3, label='step loss')
    window = 50
    if len(sft['train_loss']) > window:
        rolling = [sum(sft['train_loss'][max(0,i-window):i])/min(i,window)
                   for i in range(1, len(sft['train_loss'])+1)]
        plt.plot(rolling, label=f'rolling ({window})', linewidth=2)
    plt.xlabel('Step'); plt.ylabel('Loss')
    plt.title('SFT Training Loss'); plt.legend()
    plt.show()

# Plot KD losses
kd = results.get('kd_metrics', {})
if kd.get('train_loss'):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(kd['train_loss'], alpha=0.3)
    axes[0].set_title('Total Loss')
    axes[1].plot(kd['train_kd_loss'], alpha=0.3, color='orange')
    axes[1].set_title('KD Loss (KL)')
    axes[2].plot(kd['train_ce_loss'], alpha=0.3, color='green')
    axes[2].set_title('CE Loss')
    for ax in axes: ax.set_xlabel('Step')
    plt.tight_layout()
    plt.show()

## 4. Step-by-step (optional)

Instead of `run_pipeline`, you can run each stage separately:

In [ ]:
# # Load models
# from tt_llm.pipeline import load_model, compress_model, run_sft, run_kd, run_eval
# teacher, tok = load_model()
# student, _ = load_model()

# # Compress
# compress_model(student, tok, method=METHOD, compression_pct=COMPRESSION_PCT)

# # SFT
# run_sft(student, tok)

# # KD
# run_kd(student, teacher, tok)

# # Evaluate
# run_eval(teacher, tok, limit=EVAL_LIMIT)  # baseline
# run_eval(student, tok, limit=EVAL_LIMIT)  # compressed